# Baseline GSR do SoccerTrack v2

Clona uma versão fixa do repositório oficial, configura dados e executa os módulos oficiais de treinamento e avaliação. Para usar no Colab, selecione uma GPU e execute as células em ordem. Os dados precisam ser disponibilizados separadamente no Google Drive ou no ambiente local.

**Limitações do baseline:** treina um detector YOLO e aplica ByteTrack. O ReID padrão é um placeholder que retorna identidade desconhecida; não é uma rede ReID treinada. O código oficial usa apenas o primeiro dataset por tempo de jogo gerado para treinar, mesmo gerando outros; sua divisão de quadros não assegura validação por partida. Os resultados não garantem IDs persistentes.

O pipeline GSR precisa de vídeos panorâmicos calibrados de ambos os tempos, anotações MOT correspondentes, homografias 3×3 campo→imagem e ground truth GSR de teste. Os clipes MOT de quatro minutos e marcações de câmera locais do ProjetoPDS não são automaticamente equivalentes a esses arquivos. Não se deve inventar homografias nem combinar vídeos com anotações desalinhadas.


In [ ]:
from pathlib import Path
import os, subprocess, sys

# No Colab, monte o Drive e ajuste os caminhos abaixo. Em Jupyter local, não monte.
IS_COLAB = "google.colab" in sys.modules
if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

WORKSPACE = Path("/content" if IS_COLAB else Path.cwd()).resolve()
DATA_ROOT = Path("/content/drive/MyDrive/soccertrack-v2-data" if IS_COLAB else "./soccertrack-v2-data").resolve()
OUTPUT_ROOT = Path("/content/drive/MyDrive/soccertrack-gsr-output" if IS_COLAB else "./soccertrack-gsr-output").resolve()
REPO = WORKSPACE / "SoccerTrack-v2"
REVISION = "a752d294df2a0e816e827b70f284980698f5a50e"
print("Dados:", DATA_ROOT, "\nResultados:", OUTPUT_ROOT)


In [ ]:
if not (REPO / ".git").is_dir():
    if REPO.exists():
        raise RuntimeError(f"{REPO} existe, mas não é um clone Git; escolha outro WORKSPACE.")
    subprocess.run(["git", "clone", "https://github.com/AtomScott/SoccerTrack-v2.git", str(REPO)], check=True)
if subprocess.run(["git", "status", "--porcelain"], cwd=REPO, capture_output=True, text=True, check=True).stdout.strip():
    raise RuntimeError("O clone contém alterações locais. Use um clone limpo para preservar os arquivos.")
subprocess.run(["git", "fetch", "origin", REVISION], cwd=REPO, check=True)
subprocess.run(["git", "checkout", "--detach", REVISION], cwd=REPO, check=True)
print(subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip())


## Dependências

O projeto oficial exige Python 3.12 ou superior. A instalação pode demorar; requer acesso à internet, FFmpeg e espaço em disco. `sn-trackeval` é a variante usada para GS-HOTA.


In [ ]:
if sys.version_info < (3, 12):
    raise RuntimeError("O SoccerTrack v2 exige Python >= 3.12; selecione outro runtime.")
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO)], check=True)


In [ ]:
import shutil
import torch
import yaml
if shutil.which("ffmpeg") is None:
    raise RuntimeError("FFmpeg não encontrado no PATH.")
if not torch.cuda.is_available():
    raise RuntimeError("GPU CUDA não disponível. No Colab, selecione um ambiente com GPU.")
print("GPU:", torch.cuda.get_device_name(0))


## Configuração e dados

A configuração oficial é copiada para a pasta de resultados, sem modificar o clone. Ajuste as listas se seus dados forem diferentes e mantenha partidas de treinamento, validação e teste disjuntas. O baseline consulta ambos os tempos de cada partida selecionada.

Estrutura esperada sob `DATA_ROOT`:

```text
mot/<partida>/<1st_half|2nd_half>/gt/gt.txt
interim/<partida>/<partida>_calibrated_panorama_<1st_half|2nd_half>.mp4
interim/homography/<partida>/<partida>_homography.npy
gsr/<partida>/<partida>_<1st|2nd>.json
```

Os arquivos MOT precisam corresponder aos vídeos calibrados e à base de índices esperada pelo gerador oficial. As homografias devem ser válidas para cada vídeo calibrado; as anotações GSR de teste precisam corresponder aos mesmos frames e tempos.


In [ ]:
# Edite as listas para partidas realmente disponíveis.
TRAIN_MATCHES = [117092, 117093, 118575, 118576, 118577, 128058]
VAL_MATCHES = [118578]
TEST_MATCHES = [128057, 132831]

groups = [set(map(str, xs)) for xs in (TRAIN_MATCHES, VAL_MATCHES, TEST_MATCHES)]
if not all(groups) or any(groups[i] & groups[j] for i in range(3) for j in range(i + 1, 3)):
    raise ValueError("Informe grupos não vazios e sem partidas repetidas entre treino, validação e teste.")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
cfg = yaml.safe_load((REPO / "baselines/gsr/config.yaml").read_text())
cfg["data"].update(root=str(DATA_ROOT), train_matches=TRAIN_MATCHES, val_matches=VAL_MATCHES, test_matches=TEST_MATCHES)
cfg["mot"]["root"] = str(DATA_ROOT / "mot")
cfg["video"]["root"] = str(DATA_ROOT / "interim")
cfg["homography"]["root"] = str(DATA_ROOT / "interim/homography")
cfg["detector"].update(imgsz=1280, project=str(OUTPUT_ROOT / "detector"), dataset_dir=str(OUTPUT_ROOT / "yolo_dataset"))
cfg["eval"].update(pred_root=str(OUTPUT_ROOT / "preds"), out=str(OUTPUT_ROOT / "scores.json"))
CONFIG = OUTPUT_ROOT / "config.yaml"
CONFIG.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True))
print("Configuração:", CONFIG)


In [ ]:
# Falha antes do treinamento se faltarem entradas exigidas pelo baseline.
missing = []
for match in TRAIN_MATCHES + VAL_MATCHES + TEST_MATCHES:
    for half in ("1st_half", "2nd_half"):
        video = DATA_ROOT / "interim" / str(match) / f"{match}_calibrated_panorama_{half}.mp4"
        if not video.is_file(): missing.append(video)
        if match in TRAIN_MATCHES + VAL_MATCHES:
            mot = DATA_ROOT / "mot" / str(match) / half / "gt/gt.txt"
            if not mot.is_file(): missing.append(mot)
    if match in TEST_MATCHES:
        for p in [DATA_ROOT / "interim/homography" / str(match) / f"{match}_homography.npy",
                  *[DATA_ROOT / "gsr" / str(match) / f"{match}_{half}.json" for half in ("1st", "2nd")]]:
            if not p.is_file(): missing.append(p)
if missing:
    raise FileNotFoundError("Arquivos necessários ausentes:\n" + "\n".join(map(str, missing)))
print("Arquivos encontrados. Confirme também o alinhamento temporal e geométrico antes de treinar.")


## Treinamento e geração das predições

Esta célula invoca `python -m baselines.gsr.train --config ...` do repositório oficial. O diretório de saída pode ser sobrescrito pelo baseline em execuções repetidas: escolha outro `OUTPUT_ROOT` para cada experimento.


In [ ]:
subprocess.run([sys.executable, "-m", "baselines.gsr.train", "--config", str(CONFIG)], cwd=REPO, check=True)


## Avaliação GS-HOTA

A avaliação oficial compara posições no campo e atributos de identidade. Como o ReID padrão retorna equipe e número desconhecidos, o resultado não representa um sistema de identificação completo.


In [ ]:
subprocess.run([sys.executable, "-m", "baselines.gsr.eval", "--config", str(CONFIG)], cwd=REPO, check=True)
print((OUTPUT_ROOT / "scores.json").read_text())
